In [1]:
!pip -q install ultralytics ipywidgets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 75.1 MB/s eta 0:00:00


## 2. Cargar el modelo v2

Elige `MODEL_SOURCE`:
- `"drive"`: pesos en tu Google Drive (ajusta `DRIVE_WEIGHTS_PATH`).
- `"upload"`: te pide subir el archivo `.pt` desde tu computador.
- `"pretrained"`: usa `yolov8n.pt` solo para probar que todo funciona.

In [3]:
MODEL_SOURCE = "drive"
DRIVE_WEIGHTS_PATH = "/content/drive/MyDrive/beealarmed/models/runs/yolo11n_v2/weights/best.pt"

import os
from ultralytics import YOLO

if MODEL_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    weights = DRIVE_WEIGHTS_PATH
elif MODEL_SOURCE == "upload":
    from google.colab import files
    up = files.upload()
    weights = list(up.keys())[0]
elif MODEL_SOURCE == "pretrained":
    weights = "yolov11n.pt"
else:
    raise ValueError("MODEL_SOURCE debe ser 'drive', 'upload' o 'pretrained'")

if not os.path.exists(weights) and MODEL_SOURCE != "pretrained":
    raise FileNotFoundError(f"No encuentro los pesos en: {weights}")

model = YOLO(weights)
print("Modelo cargado:", weights)
print("Clases:", model.names)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Modelo cargado: /content/drive/MyDrive/beealarmed/models/runs/yolo11n_v2/weights/best.pt
Clases: {0: 'bee', 1: 'varroa', 2: 'wasp'}


## Upload images for detection

In [5]:
import os, shutil
from google.colab import files

IMG_DIR = "/content/imagenes"
os.makedirs(IMG_DIR, exist_ok=True)

uploaded = files.upload()
for name in uploaded.keys():
    shutil.move(name, os.path.join(IMG_DIR, name))

valid_ext = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
image_files = sorted(f for f in os.listdir(IMG_DIR) if f.lower().endswith(valid_ext))
print(f"{len(image_files)} imagen(es) disponibles:", image_files)

Saving 2017-10-17_1-39-36.mp4-bee_id_3563-2670-1.png to 2017-10-17_1-39-36.mp4-bee_id_3563-2670-1.png
Saving 2017-10-17_1-39-36.mp4-bee_id_3564-2700-1.png to 2017-10-17_1-39-36.mp4-bee_id_3564-2700-1.png
Saving 2017-10-17_1-39-36.mp4-bee_id_3665-5475-1.png to 2017-10-17_1-39-36.mp4-bee_id_3665-5475-1.png
3 imagen(es) disponibles: ['2017-10-17_1-39-36.mp4-bee_id_3563-2670-1.png', '2017-10-17_1-39-36.mp4-bee_id_3564-2700-1.png', '2017-10-17_1-39-36.mp4-bee_id_3665-5475-1.png']


## Interactive visor with confidence slider


In [6]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

BASE_CONF = 0.01
_cache = {}

PALETTE = [
    (255, 56, 56), (255, 157, 151), (255, 112, 31), (255, 178, 29), (207, 210, 49),
    (72, 249, 10), (146, 204, 23), (61, 219, 134), (26, 147, 52), (0, 212, 187),
    (44, 153, 168), (0, 194, 255), (52, 69, 147), (100, 115, 255), (0, 24, 236),
    (132, 56, 255), (82, 0, 133), (203, 56, 255), (255, 149, 200), (255, 55, 199),
]

def get_result(name):
    if name not in _cache:
        path = os.path.join(IMG_DIR, name)
        _cache[name] = model.predict(path, conf=BASE_CONF, verbose=False)[0]
    return _cache[name]

def draw_detections(result, thr):
    img = result.orig_img.copy()  # BGR
    h, w = img.shape[:2]
    thickness = max(2, int(round(min(h, w) / 300)))
    font_scale = max(0.5, min(h, w) / 900)
    counts = {}

    if result.boxes is not None:
        for b in result.boxes:
            conf = float(b.conf[0])
            if conf < thr:
                continue
            cls = int(b.cls[0])
            name = result.names[cls]
            counts[name] = counts.get(name, 0) + 1

            x1, y1, x2, y2 = map(int, b.xyxy[0].tolist())
            color = PALETTE[cls % len(PALETTE)]
            cv2.rectangle(img, (x1, y1), (x2, y2), color, thickness)

            label = f"{name} {conf:.2f}"
            (tw, th), bl = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, font_scale, 1)
            y_top = max(y1 - th - bl - 4, 0)
            cv2.rectangle(img, (x1, y_top), (x1 + tw + 4, y_top + th + bl + 4), color, -1)
            cv2.putText(img, label, (x1 + 2, y_top + th + 1),
                        cv2.FONT_HERSHEY_SIMPLEX, font_scale, (255, 255, 255), 1, cv2.LINE_AA)

    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB), counts

# --- Widgets ---
img_dropdown = widgets.Dropdown(
    options=image_files, description="Imagen:",
    layout=widgets.Layout(width="60%"),
)
conf_slider = widgets.FloatSlider(
    value=0.25, min=0.01, max=0.99, step=0.01,
    description="Confianza:", continuous_update=False,
    readout_format=".2f", layout=widgets.Layout(width="60%"),
)
out = widgets.Output()

def render(*_):
    with out:
        clear_output(wait=True)
        if not img_dropdown.value:
            print("No hay imágenes. Ejecuta la celda de subida primero.")
            return
        result = get_result(img_dropdown.value)
        rgb, counts = draw_detections(result, conf_slider.value)

        plt.figure(figsize=(10, 8))
        plt.imshow(rgb)
        plt.axis("off")
        total = sum(counts.values())
        plt.title(f"{img_dropdown.value}  |  umbral ≥ {conf_slider.value:.2f}  |  {total} detección(es)")
        plt.show()

        if counts:
            print("Detecciones por clase:")
            for k, v in sorted(counts.items(), key=lambda kv: -kv[1]):
                print(f"  - {k}: {v}")
        else:
            print("Sin detecciones con este umbral. Prueba bajando el slider.")

img_dropdown.observe(render, names="value")
conf_slider.observe(render, names="value")

display(widgets.VBox([img_dropdown, conf_slider, out]))
render()

## Optional: saving annotated image

In [ ]:
import cv2, os
from google.colab import files

result = get_result(img_dropdown.value)
rgb, counts = draw_detections(result, conf_slider.value)

out_path = f"/content/resultado_{os.path.splitext(img_dropdown.value)[0]}_conf{conf_slider.value:.2f}.jpg"
cv2.imwrite(out_path, cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
print("Guardada en:", out_path)
files.download(out_path)